# Lab 05 — Fixing TFIM Simulation Limitations

This notebook identifies and fixes five concrete limitations from the TFIM pipeline of Challenge 3 (Quantathon CR 2026).  
Each section is self-contained and can be run independently.

| § | Limitation | Fix strategy |
|---|---|---|
| §1 | Bug: ∆t = 0.05 violates the 5 % error threshold at h/J = 2 | Adaptive ∆t selection based on commutator error bound |
| §2 | Gap E₁−E₀ may be the parity doublet, not the bulk excitation | Compute k=3 eigenstates and test exponential vs. power-law scaling |
| §3 | Phase transition rounded for N = 6–10 | Binder cumulant + thermodynamic-limit extrapolation |
| §4 | Linear ZNE with two points over-corrects at long times | Richardson extrapolation + model comparison |
| §5 | "2×2 periodic" lattice is a 4-site ring, not truly 2D periodic | Bond-duplication check; 3×4 is the smallest genuinely periodic lattice |

**Requirements:** `numpy`, `scipy`, `matplotlib` only — no Qiskit or hardware access needed.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidggonz/Qiskit-Quantum-Computing---UOttawa-Fall-Fest-26/blob/main/Lab_05-TFIM_Limitations_Fix.ipynb)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Environment ready.')

In [ ]:
# ── Shared math core ──────────────────────────────────────────────────────────
# Minimal re-implementation of pauli_ops.py so this notebook is self-contained.

_P = {
    'x': sp.csr_matrix(np.array([[0, 1], [1, 0]], dtype=complex)),
    'z': sp.csr_matrix(np.array([[1, 0], [0, -1]], dtype=complex)),
}
_I2 = sp.eye(2, format='csr', dtype=complex)


def pauli(n: int, N: int, kind: str):
    """Return the `kind` Pauli operator acting on qubit n in an N-qubit chain."""
    ops = [_P[kind] if i == n else _I2 for i in range(N)]
    result = ops[0]
    for op in ops[1:]:
        result = sp.kron(result, op, format='csr')
    return result


def build_tfim(N: int, J: float = 1.0, h: float = 1.0):
    """H = -J Σᵢ ZᵢZᵢ₊₁  -  h Σᵢ Xᵢ  (periodic chain)."""
    dim = 2 ** N
    H = sp.csr_matrix((dim, dim), dtype=complex)
    for i in range(N):
        j = (i + 1) % N
        H -= J * pauli(i, N, 'z') @ pauli(j, N, 'z')
    for i in range(N):
        H -= h * pauli(i, N, 'x')
    return H.real


def observables(N):
    """Return (Mz², Mx, Mzz, Mz⁴) as sparse matrices."""
    Zs = [pauli(i, N, 'z') for i in range(N)]
    Xs = [pauli(i, N, 'x') for i in range(N)]
    Mz  = sum(Zs)
    Mx  = sum(Xs)
    Mzz = sum(Zs[i] @ Zs[(i + 1) % N] for i in range(N))
    Mz2 = Mz @ Mz
    Mz4 = Mz2 @ Mz2
    return Mz2, Mx, Mzz, Mz4


def ev(psi, op):
    """Expectation value <psi|op|psi>."""
    return np.real(psi.conj() @ (op @ psi))


def diag_gs(N, h, J=1.0, k=1):
    """Diagonalize H and return the k lowest eigenvalues/eigenvectors."""
    H = build_tfim(N, J=J, h=h)
    vals, vecs = spla.eigsh(H, k=k, which='SA')
    idx = np.argsort(vals)
    return vals[idx], vecs[:, idx]


print('Math core ready.')

---
## §1 — Limitation 3: ∆t = 0.05 violates the 5 % threshold at h/J = 2

**Problem:** With `dt = 0.05` the Trotter error on C_ZZ reaches 8.46 % for h/J = 2 (second-order decomposition).  
The second-order Trotter error scales as **O(∆t²)**: halving ∆t reduces the error ~4×, from 8.46 % to ~2.10 %.

**Why h/J = 2 is the worst case:**  
The leading error term is proportional to `||[H_ZZ, H_X]||`. This commutator is largest when neither term dominates — precisely near h/J ~ 1–2, where both ZZ and X contributions are comparable.

**Fix:** A function that selects ∆t for a given h/J so that the accumulated Trotter error stays below a specified threshold.

In [ ]:
# ── §1: Trotter error as a function of dt ─────────────────────────────────────

def trotter_error_bound(N: int, h: float, J: float, dt: float) -> float:
    """
    Error bound for one step of second-order Trotter:
        ||[H_ZZ, H_X]|| * dt² / 12

    Multiply by the number of steps to get the accumulated bound for fixed T.
    Uses the Frobenius norm (cheap for sparse matrices; overestimates spectral
    norm but sufficient for comparing regimes).

    Reference: Suzuki 1976, eq. (2.5).
    """
    H_ZZ = build_tfim(N, J=J, h=0.0)   # ZZ term only
    H_X  = build_tfim(N, J=0.0, h=h)   # X term only
    comm = H_ZZ @ H_X - H_X @ H_ZZ
    norm = sp.linalg.norm(comm, 'fro')
    return (norm * dt**2) / 12.0


def recommend_dt(
    N: int,
    h: float,
    J: float = 1.0,
    T: float = 20.0,
    threshold: float = 0.05,
    dt_min: float = 0.005,
    dt_max: float = 0.1,
) -> float:
    """
    Return the largest dt in [dt_min, dt_max] such that the accumulated
    Trotter error bound over total time T stays below `threshold`.
    """
    dt = dt_max
    while dt >= dt_min:
        steps = int(T / dt)
        if trotter_error_bound(N, h, J, dt) * steps <= threshold:
            return round(dt, 5)
        dt /= 2.0
    return dt_min


# --- Demonstration for the three field values used in the original paper ---
N_demo, T_demo = 6, 20.0

print(f"{'h/J':>6} | {'dt=0.05 bound':>14} | {'dt=0.025 bound':>15} | {'recommended dt':>14}")
print("-" * 60)
for h in [0.5, 1.0, 2.0]:
    e05  = trotter_error_bound(N_demo, h, 1.0, 0.05)  * int(T_demo / 0.05)
    e025 = trotter_error_bound(N_demo, h, 1.0, 0.025) * int(T_demo / 0.025)
    dt_r = recommend_dt(N_demo, h, T=T_demo)
    f05  = "FAIL" if e05  > 0.05 else "ok"
    f025 = "FAIL" if e025 > 0.05 else "ok"
    print(f"{h:>6.1f} | {e05:>10.4f}  {f05:>4} | {e025:>11.4f}  {f025:>4} | {dt_r:>14.4f}")

In [ ]:
# ── Plot: error bound vs dt for all three field values ────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
dts = np.logspace(-2, -0.7, 40)
colors = ['#1f77b4', '#ff7f0e', '#d62728']

for h, color in zip([0.5, 1.0, 2.0], colors):
    errs = [trotter_error_bound(6, h, 1.0, dt) * int(20.0 / dt) for dt in dts]
    ax.loglog(dts, errs, color=color, label=f'h/J = {h}')

ax.axhline(0.05, color='black', ls='--', lw=1.2, label='5 % threshold')
ax.axvline(0.05,  color='gray',  ls=':',  lw=1.0, label='dt = 0.05 (original)')
ax.axvline(0.025, color='gray',  ls='-.', lw=1.0, label='dt = 0.025 (reference)')
ax.loglog(dts, dts**2 * 1.5e3, 'k--', lw=0.8, alpha=0.4, label='slope O(∆t²)')

ax.set_xlabel('∆t')
ax.set_ylabel('Accumulated Trotter error bound (T = 20)')
ax.set_title('§1 — Adaptive ∆t selection by h/J')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('fig_dt_selection.png', dpi=120)
plt.show()

print("Conclusion: h/J = 2 requires dt ≤ 0.025 to meet the 5 % threshold.")
print("h/J = 0.5 and 1.0 already pass with dt = 0.05 because one term dominates.")

---
## §2 — Limitation 2: Is E₁ − E₀ the parity doublet or the true bulk gap?

**Problem:** For h/J < 1 (ferromagnetic phase), the periodic TFIM has a **symmetry-protected doublet** in the thermodynamic limit: two states with opposite Z₂ parity whose splitting E₁ − E₀ vanishes *exponentially fast* as N → ∞. If the code only requests `k=1` eigenstate and then inspects E₁ − E₀ from a separate call, it may be reporting this doublet gap instead of the actual bulk excitation (which closes as a power law at the critical point).

**Fix:** Always compute `k = 3` eigenstates and diagnose the gaps by their N-scaling.

In [ ]:
# ── §2: Three-level spectrum analysis ─────────────────────────────────────────

def spectrum_analysis(N_list, h_list, J=1.0, k=3):
    """
    Compute the k lowest eigenvalues and classify the two lowest gaps.
    Returns dict {(N, h): {'E': array(k), 'gap01': E1-E0, 'gap02': E2-E0}}.
    """
    results = {}
    for N in N_list:
        for h in h_list:
            vals, _ = diag_gs(N, h, J=J, k=k)
            results[(N, h)] = {
                'E': vals,
                'gap01': vals[1] - vals[0],   # possible parity doublet
                'gap02': vals[2] - vals[0],   # first bulk excitation above doublet
            }
    return results


N_list_gap = [4, 6, 8, 10]
h_list_gap = [0.5, 1.0, 2.0]
print('Computing k=3 spectrum...')
spec = spectrum_analysis(N_list_gap, h_list_gap, k=3)

print(f"\n{'N':>4} | {'h/J':>5} | {'E0/N':>10} | {'gap01 (E1-E0)':>14} | {'gap02 (E2-E0)':>14} | Diagnosis")
print("-" * 82)
for N in N_list_gap:
    for h in h_list_gap:
        r = spec[(N, h)]
        g01, g02 = r['gap01'], r['gap02']
        ratio = g01 / g02 if g02 > 1e-12 else float('nan')
        if h < 1.0 and ratio < 0.3:
            diag = "WARNING: doublet — use E2-E0 as bulk gap"
        elif h == 1.0:
            diag = "~critical"
        else:
            diag = "OK: paramagnetic"
        print(f"{N:>4} | {h:>5.1f} | {r['E'][0]/N:>10.5f} | {g01:>14.6f} | {g02:>14.6f} | {diag}")

In [ ]:
# ── Gap scaling with N: exponential (doublet) vs power-law (bulk) ─────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
Ns = np.array([4, 6, 8, 10])

for ax, h in zip(axes, [0.5, 1.0, 2.0]):
    g01s = [spec[(N, h)]['gap01'] for N in Ns]
    g02s = [spec[(N, h)]['gap02'] for N in Ns]

    ax.semilogy(Ns, g01s, 'o-', label='E₁−E₀  (parity doublet?)')
    ax.semilogy(Ns, g02s, 's--', label='E₂−E₀  (bulk gap)')

    # Exponential fit for E1-E0 in the ordered phase
    if h < 1.0:
        coeffs = np.polyfit(Ns, np.log(np.array(g01s) + 1e-15), 1)
        Nfit = np.linspace(Ns[0], Ns[-1], 100)
        ax.semilogy(Nfit, np.exp(np.polyval(coeffs, Nfit)),
                    'b:', alpha=0.5, label=f'e^({coeffs[0]:.2f}·N) fit')

    ax.set_xlabel('N')
    ax.set_ylabel('Gap')
    ax.set_title(f'h/J = {h}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('§2 — Parity doublet vs. bulk excitation', fontsize=12)
plt.tight_layout()
plt.savefig('fig_gap_scaling.png', dpi=120)
plt.show()

print("Takeaway:")
print("  h/J = 0.5 (ferromagnetic): E1-E0 decays exponentially → parity doublet.")
print("             Report E2-E0 as the bulk excitation gap.")
print("  h/J = 1.0 (critical):      both gaps close as ~1/N (1D Ising universality).")
print("  h/J = 2.0 (paramagnetic):  E1-E0 ≈ E2-E0; symmetry broken, no doublet.")

---
## §3 — Limitation 1: Phase transition rounded for N = 6–10

**Problem:** Finite-size effects smooth out the singularity of the phase transition. The magnetization and susceptibility are rounded, making it hard to extract the critical point h_c from a single system size.

**Fix:** The **Binder cumulant**
$$U_4 = 1 - \frac{\langle M_z^4 \rangle}{3\,\langle M_z^2 \rangle^2}$$
has the property that its curves for different N **cross at exactly the critical point**, regardless of the rounding. Extrapolating the crossing positions vs. 1/N recovers h_c in the thermodynamic limit without any finite-size scaling hypothesis.

In [ ]:
# ── §3: Binder cumulant ───────────────────────────────────────────────────────

def binder_cumulant(N: int, h: float, J: float = 1.0) -> float:
    """
    U₄ = 1 - <Mz⁴> / (3 <Mz²>²)

    Limits:
      U₄ → 2/3 ≈ 0.667   in the ordered phase (N → ∞)
      U₄ → 0              in the disordered phase
      U₄ ≈ 0.47           at the critical point (universal, N-independent)
    """
    Mz2_op, _, _, Mz4_op = observables(N)
    _, vecs = diag_gs(N, h, J=J, k=1)
    gs = vecs[:, 0]
    m2 = ev(gs, Mz2_op) / N**2
    m4 = ev(gs, Mz4_op) / N**4
    return 1.0 - m4 / (3.0 * m2**2) if m2 > 1e-14 else 0.0


N_list_binder = [4, 6, 8, 10]
h_values_binder = np.linspace(0.3, 1.7, 30)  # fine sweep around h/J = 1

print('Computing Binder cumulant...')
binder_data = {N: [] for N in N_list_binder}
for N in N_list_binder:
    for h in h_values_binder:
        binder_data[N].append(binder_cumulant(N, h))
    print(f'  N={N} done')
print('Done.')

In [ ]:
# ── Plot: Binder cumulant + critical-point extrapolation ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_N = plt.cm.viridis(np.linspace(0.15, 0.85, len(N_list_binder)))

# Left panel: U4 vs h/J
ax = axes[0]
for N, color in zip(N_list_binder, colors_N):
    ax.plot(h_values_binder, binder_data[N], '-o', ms=4, color=color, label=f'N={N}')
ax.axvline(1.0, color='red',  ls='--', lw=1.5, label='h/J = 1 (exact)')
ax.axhline(2/3, color='gray', ls=':', lw=1.0, label='U₄ = 2/3 (ordered)')
ax.set_xlabel('h/J')
ax.set_ylabel('Binder cumulant U₄')
ax.set_title('§3 — Binder cumulant: crossing at h_c')
ax.legend()
ax.grid(True, alpha=0.3)

# Right panel: crossing positions vs 1/N → thermodynamic limit
ax2 = axes[1]
crossings, N_pairs = [], []
for i in range(len(N_list_binder) - 1):
    N1, N2 = N_list_binder[i], N_list_binder[i + 1]
    diff = np.array(binder_data[N1]) - np.array(binder_data[N2])
    for idx in np.where(np.diff(np.sign(diff)))[0]:
        h1, h2 = h_values_binder[idx], h_values_binder[idx + 1]
        d1, d2 = diff[idx], diff[idx + 1]
        h_cross = h1 - d1 * (h2 - h1) / (d2 - d1)
        crossings.append(h_cross)
        N_pairs.append((N1 + N2) / 2)

if crossings:
    ax2.scatter(N_pairs, crossings, s=80, zorder=5, color='steelblue', label='Binder crossings')
    if len(crossings) >= 2:
        inv_N = [1 / n for n in N_pairs]
        p = np.polyfit(inv_N, crossings, 1)
        x_fit = np.linspace(0, max(inv_N) * 1.05, 100)
        ax2.plot(x_fit, np.polyval(p, x_fit), 'r--',
                 label=f'Linear extrap.: h_c = {p[1]:.4f}')
        ax2.scatter([0], [p[1]], s=130, color='red', zorder=6,
                    label=f'Extrapolated h_c = {p[1]:.4f}')
    ax2.axhline(1.0, color='gray', ls=':', label='h/J = 1 (exact)')
ax2.set_xlabel('1/N_avg of pair')
ax2.set_ylabel('Crossing h_c')
ax2.set_title('Thermodynamic-limit extrapolation (1/N → 0)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_binder_cumulant.png', dpi=120)
plt.show()

if crossings:
    hc = np.polyfit([1/n for n in N_pairs], crossings, 1)[1]
    print(f'Crossing pairs (N_avg, h_c): {list(zip(N_pairs, [round(c,4) for c in crossings]))}')
    print(f'Extrapolated h_c = {hc:.4f}  (exact: 1.0000)')

---
## §4 — Limitation 5: Linear ZNE with two points over-corrects at long times

**Problem:**  
- **Linear extrapolation** with λ = {1, 3, 5} assumes the error grows linearly with the fold factor. At large circuit depths or long simulation times the noise response saturates or has quadratic curvature, so the linear model over-shoots.  
- **Richardson extrapolation** is exact when the error has a single dominant term of order p: it uses exact Lagrange weights to cancel that term.  
- **Quadratic regression** captures curvature but needs at least three points (already available: λ = 1, 3, 5).

**Fix:** Implement all three models, propagate errors, and compare residuals.

In [ ]:
# ── §4: ZNE extrapolators ─────────────────────────────────────────────────────

def richardson_extrapolate(lambdas, values, errors=None, order=1):
    """
    Richardson extrapolation of given `order`.
    Cancels the O(λ^order) error term using `order+1` points.

    Weights are Lagrange interpolation coefficients evaluated at λ=0:
        w_i = ∏_{j≠i} (−λ_j) / (λ_i − λ_j)

    Reference: Temme, Bravyi & Gambetta, PRL 119, 180509 (2017).
    """
    lam = np.asarray(lambdas[:order + 1], dtype=float)
    y   = np.asarray(values[:order + 1],  dtype=float)

    w = np.ones(order + 1)
    for i in range(order + 1):
        for j in range(order + 1):
            if i != j:
                w[i] *= -lam[j] / (lam[i] - lam[j])

    val = float(w @ y)
    err = float(np.sqrt((w**2) @ (np.asarray(errors[:order + 1])**2))) \
          if errors is not None else float('nan')
    return val, err


def zne_polyfit(lambdas, values, errors=None, deg=1):
    """Weighted polynomial fit of degree `deg`, evaluated at λ=0."""
    x = np.asarray(lambdas, dtype=float)
    y = np.asarray(values, dtype=float)
    w = 1.0 / np.asarray(errors, dtype=float) if errors is not None and all(
        e > 0 for e in errors) else None
    coeffs, cov = np.polyfit(x, y, deg, w=w,
                              cov='unscaled' if w is not None else True)
    return float(coeffs[-1]), float(np.sqrt(cov[-1, -1]))


rng = np.random.default_rng(42)
lambdas = [1, 3, 5]
y_true  = 0.72
noise   = 0.02

# Case A: truly linear noise model (shallow circuits)
y_lin  = [y_true - 0.05*(l - 1) + rng.normal(0, noise) for l in lambdas]
# Case B: quadratic noise model (deep circuits / long times)
y_quad = [y_true - 0.02*(l - 1) - 0.008*(l - 1)**2 + rng.normal(0, noise) for l in lambdas]

for case, y_data, label in [('A — linear noise', y_lin, 'linear'), ('B — quadratic noise', y_quad, 'quad')]:
    sigma = [noise] * 3
    r1, e1 = zne_polyfit(lambdas, y_data, sigma, deg=1)
    r2, e2 = zne_polyfit(lambdas, y_data, sigma, deg=2)
    rR1, eR1 = richardson_extrapolate(lambdas, y_data, sigma, order=1)
    rR2, eR2 = richardson_extrapolate(lambdas, y_data, sigma, order=2)
    print(f'\n--- Case {case} ---')
    print(f'  True value:            {y_true:.4f}')
    print(f'  Linear  (deg=1):       {r1:.4f} ± {e1:.4f}   abs err: {abs(r1-y_true):.4f}')
    print(f'  Quadratic (deg=2):     {r2:.4f} ± {e2:.4f}   abs err: {abs(r2-y_true):.4f}')
    print(f'  Richardson (order=1):  {rR1:.4f} ± {eR1:.4f}   abs err: {abs(rR1-y_true):.4f}')
    print(f'  Richardson (order=2):  {rR2:.4f} ± {eR2:.4f}   abs err: {abs(rR2-y_true):.4f}')

In [ ]:
# ── Plot: visual comparison of extrapolators ──────────────────────────────────
rng2 = np.random.default_rng(42)
y_lin2  = [y_true - 0.05*(l - 1) + rng2.normal(0, noise) for l in lambdas]
rng2 = np.random.default_rng(42)
y_quad2 = [y_true - 0.02*(l - 1) - 0.008*(l - 1)**2 + rng2.normal(0, noise) for l in lambdas]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (y_data, title, true_fn) in zip(axes, [
    (y_lin2,  'Case A: linear noise model',    lambda l: y_true - 0.05*(l-1)),
    (y_quad2, 'Case B: quadratic noise model', lambda l: y_true - 0.02*(l-1) - 0.008*(l-1)**2),
]):
    sigma = [noise] * 3
    l_ext = np.linspace(0, 5.8, 100)

    ax.errorbar(lambdas, y_data, yerr=sigma, fmt='ko', ms=7, capsize=4,
                zorder=5, label='Measurements')
    ax.plot(l_ext, [true_fn(l) for l in l_ext], 'k--', lw=1,
            label='True noise curve (unknown)')
    ax.axvline(0, color='gray', ls=':', lw=0.8)
    ax.axhline(y_true, color='gray', ls=':', lw=0.8)

    for deg, color, lbl in [(1, '#1f77b4', 'Linear (deg=1)'),
                             (2, '#ff7f0e', 'Quadratic (deg=2)')]:
        c = np.polyfit(lambdas, y_data, deg)
        ax.plot(l_ext, np.polyval(c, l_ext), color=color, lw=1.5, label=lbl)
        v, e = zne_polyfit(lambdas, y_data, sigma, deg=deg)
        ax.errorbar(0, v, yerr=e, fmt='s', color=color, ms=9, capsize=5, zorder=6)

    rV, rE = richardson_extrapolate(lambdas, y_data, sigma, order=1)
    ax.errorbar(-0.3, rV, yerr=rE, fmt='^', color='green', ms=9, capsize=5,
                zorder=6, label='Richardson ord=1')
    ax.scatter([0], [y_true], marker='*', s=200, color='red', zorder=7,
               label=f'True value = {y_true}')

    ax.set_xlim(-0.9, 6.0)
    ax.set_xlabel('Fold factor λ')
    ax.set_ylabel('Observable')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('§4 — ZNE: extrapolator comparison', fontsize=12)
plt.tight_layout()
plt.savefig('fig_zne_comparison.png', dpi=120)
plt.show()

print('\nPractical recommendation:')
print('  Shallow circuits (T·Γ < 1):  linear is sufficient.')
print('  Deep circuits (T·Γ > 1):     use quadratic or Richardson order=2.')
print('  Always save raw (λᵢ, yᵢ, σᵢ) per observable to allow re-fitting later.')

---
## §5 — Limitation 7: "2×2 periodic" lattice is a 4-site ring

**Problem:** When L = 2 along one direction, the wrap-around bond connects the **same pair of sites** that the ordinary nearest-neighbor bond already connects. The result is a doubled hopping amplitude on those bonds, not a new bond. This makes it topologically identical to the open-boundary case — not a torus.

**Fix:** Any dimension must be **L ≥ 3** for wrap-around to add a genuinely new bond. The smallest lattice that is truly periodic in both directions and fits on the Quantinuum H2 (≤ 56 qubits) is **3×4 = 24 qubits**.

In [ ]:
# ── §5: Bond-duplication check ────────────────────────────────────────────────

def hubbard_bond_list(Lx: int, Ly: int):
    """
    Return a dict {(site_i, site_j): count} for the periodic Fermi-Hubbard
    lattice. count > 1 means the same bond is added twice (wrap-around bug).
    """
    sites = {(x, y): y * Lx + x for y in range(Ly) for x in range(Lx)}
    bonds = {}
    for x in range(Lx):
        for y in range(Ly):
            for (dx, dy) in [(1, 0), (0, 1)]:
                xn, yn = (x + dx) % Lx, (y + dy) % Ly
                s1, s2 = sites[(x, y)], sites[(xn, yn)]
                key = (min(s1, s2), max(s1, s2))
                bonds[key] = bonds.get(key, 0) + 1
    return bonds


print(f"{'Geometry':>10} | {'Sites':>6} | {'Qubits':>6} | {'Bonds':>6} | {'Duplicates':>10} | Verdict")
print("-" * 70)
for (Lx, Ly) in [(2, 2), (2, 3), (2, 4), (3, 3), (3, 4)]:
    bonds = hubbard_bond_list(Lx, Ly)
    n_dup = sum(1 for w in bonds.values() if w > 1)
    verdict = "PASS" if n_dup == 0 else f"FAIL ({n_dup} duplicated bonds)"
    print(f"{Lx}×{Ly:>7}  | {Lx*Ly:>6} | {2*Lx*Ly:>6} | {len(bonds):>6} | {n_dup:>10} | {verdict}")

print()
print('3×4 (24 qubits) is the smallest genuinely 2D periodic lattice that fits H2.')
print('For any geometry with L=2 in one direction, use open boundaries or describe')
print('it correctly as a ring/ladder, not a periodic 2D lattice.')

In [ ]:
# ── Plot: bond diagrams for three geometries ─────────────────────────────────
from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (Lx, Ly), title in zip(
    axes,
    [(2, 2), (2, 3), (3, 4)],
    ['2×2  (4-site ring; FAIL)', '2×3  (FAIL)', '3×4  (genuinely periodic; PASS)'],
):
    bonds = hubbard_bond_list(Lx, Ly)
    pos = {y * Lx + x: (x, y) for y in range(Ly) for x in range(Lx)}

    for (s1, s2), weight in bonds.items():
        x1, y1 = pos[s1]
        x2, y2 = pos[s2]
        # Detect wrap-around bonds by jump size
        dx, dy = abs(x2 - x1), abs(y2 - y1)
        wrap = (dx > 1) or (dy > 1)
        color = 'red' if (wrap and weight > 1) else ('royalblue' if wrap else 'black')
        ls = '--' if wrap else '-'
        lw = 1.5 + weight * 0.8
        ax.plot([x1, x2], [y1, y2], color=color, lw=lw, ls=ls, alpha=0.75)

    for s, (x, y) in pos.items():
        ax.scatter(x, y, s=160, color='steelblue', zorder=5)
        ax.annotate(str(s), (x, y), xytext=(5, 5),
                    textcoords='offset points', fontsize=8)

    ax.set_title(title)
    ax.set_xlim(-0.5, Lx - 0.3)
    ax.set_ylim(-0.5, Ly - 0.3)
    ax.set_aspect('equal')
    ax.axis('off')

legend_els = [
    Line2D([0], [0], color='black',    lw=1.5, label='Ordinary bond'),
    Line2D([0], [0], color='royalblue',lw=1.5, ls='--', label='Wrap-around (OK)'),
    Line2D([0], [0], color='red',      lw=3.0, ls='--', label='Wrap-around DUPLICATED'),
]
fig.legend(handles=legend_els, loc='lower center', ncol=3,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('§5 — Periodicity in the 2D Fermi-Hubbard lattice', fontsize=12)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('fig_lattice_periodicity.png', dpi=120)
plt.show()

---
## Summary

| § | Limitation | Effort | What was implemented |
|---|---|---|---|
| §1 | ∆t = 0.05 fails at h/J = 2 | < 1 h | `recommend_dt()` selects dt per h/J from the commutator error bound |
| §2 | E₁−E₀ may be the parity doublet | < 1 h | `diag_gs(k=3)` + exponential vs. power-law gap scaling test |
| §3 | Transition rounded for small N | ~ 2 h | Binder cumulant U₄ + 1/N extrapolation of h_c |
| §4 | Linear ZNE over-corrects | ~ 2 h | Richardson extrapolation (orders 1 & 2) with error propagation |
| §5 | 2×2 = ring, not a torus | < 30 min | Bond-counting check; 3×4 identified as the smallest valid 2D lattice |

### Limitations that cannot be fixed quickly

| Limitation | Why |
|---|---|
| Lim. 4 (few shots) | Requires H2 budget or long local emulation; not a code bug |
| Lim. 6 (VQE spin contamination) | Needs symmetry projector + new ansatz; ~1 day |
| Lim. 8 (Iceberg: single case) | Requires depth sweep on hardware; Nexus cost |
| Lim. 9 (no quantum advantage) | Not a bug; needs honest classical benchmark (MPS/DMRG) |
| Lim. 10 (Nexus queue) | External infrastructure |

### Suggested integration back into the original codebase

- **§1:** Replace the fixed `QUENCH_DT = 0.05` in `config.py` with a call to `recommend_dt(N, h, T=QUENCH_STEPS*QUENCH_DT)` per h value.
- **§2:** Change `eigsh(H, k=1, ...)` in `exact_diagonalization.py` to `k=3` and expose `gap01`/`gap02` in the output dict.
- **§3:** Add `binder_cumulant()` as a new section in `run_ed.py`.
- **§4:** Add `richardson_extrapolate()` to `zne_fit.py` alongside `zne_extrapolate()` and expose a `model` parameter.
- **§5:** Add the bond-check assertion to `HubbardLattice.__init__` in `fh_lattice.py`.